# 02 — Enriquecimiento y Unificación

Este notebook explora los datos RAW en Snowflake, muestra la integración con Taxi Zones,
la unificación de esquemas Yellow/Green y la normalización de catálogos (vendor, payment_type, rate_code).

**Prerequisito:** Haber ejecutado `01_ingest_raw.ipynb` (datos en `NYC_TAXI_P3.RAW`).

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("NYC_Taxi_Enrichment").getOrCreate()

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

YEAR_START   = int(os.environ["YEAR_START"])
YEAR_END     = int(os.environ["YEAR_END"])
SERVICES     = os.environ["SERVICES"].split(",")

print(f"Servicios: {SERVICES}")
print(f"Rango: {YEAR_START}–{YEAR_END}")

Servicios: ['yellow', 'green']
Rango: 2015–2025


## 2.1 Crear tabla TRIPS_ENRICHED

Aplica la integración de Taxi Zones, unificación de Yellow/Green y normalización de catálogos creando una nueva tabla en `RAW`.

In [3]:
from py4j.java_gateway import java_import
java_import(spark._jvm, "net.snowflake.spark.snowflake.Utils")
SfUtils = spark._jvm.net.snowflake.spark.snowflake.Utils

print("=" * 60)
print("CELDA 2: Crear tabla TRIPS_ENRICHED en RAW")
print("=" * 60)

ctas_sql = """
CREATE OR REPLACE TABLE NYC_TAXI_P3.RAW.TRIPS_ENRICHED AS
SELECT
    -- Unificación
    COALESCE(r.tpep_pickup_datetime,  r.lpep_pickup_datetime)  AS pickup_datetime,
    COALESCE(r.tpep_dropoff_datetime, r.lpep_dropoff_datetime) AS dropoff_datetime,
    
    r.PULocationID AS pu_location_id,
    pz.Zone                               AS pu_zone,
    pz.Borough                            AS pu_borough,
    r.DOLocationID AS do_location_id,
    dz.Zone                               AS do_zone,
    dz.Borough                            AS do_borough,

    r.service_type,
    r.VendorID AS vendor_id,
    CASE r.VendorID
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'VeriFone Inc.'
        ELSE 'Unknown'
    END                                   AS vendor_name,
    r.RatecodeID::INTEGER                 AS rate_code_id,
    CASE r.RatecodeID::INTEGER
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau/Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END                                   AS rate_code_desc,
    r.payment_type,
    CASE r.payment_type
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Other'
    END                                   AS payment_type_desc,
    r.trip_type,

    r.passenger_count,
    r.trip_distance,
    r.store_and_fwd_flag,

    r.fare_amount,
    r.extra,
    r.mta_tax,
    r.tip_amount,
    r.tolls_amount,
    r.improvement_surcharge,
    r.congestion_surcharge,
    r.Airport_fee AS airport_fee,
    r.total_amount,

    r.run_id,
    r.ingested_at_utc,
    r.source_year,
    r.source_month
FROM NYC_TAXI_P3.RAW.TRIPS_RAW r
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP pz ON r.PULocationID = pz.LocationID
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP dz ON r.DOLocationID = dz.LocationID
"""

print(">>> Ejecutando CTAS en Snowflake (server-side)...")
SfUtils.runQuery(SF_OPTIONS, ctas_sql)
print("✓ TRIPS_ENRICHED creada exitosamente en RAW")
print("=" * 60)

CELDA 2: Crear tabla TRIPS_ENRICHED en RAW
>>> Ejecutando CTAS en Snowflake (server-side)...
✓ TRIPS_ENRICHED creada exitosamente en RAW


## 2.2 Verificación de la tabla enriquecida

In [4]:
print("=" * 60)
print("CELDA 3: Verificación")
print("=" * 60)
df_sample = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", "SELECT * FROM NYC_TAXI_P3.RAW.TRIPS_ENRICHED LIMIT 5").load())
df_sample.show(truncate=False)

CELDA 3: Verificación
+-------------------+-------------------+--------------+-------------------+----------+--------------+---------------------+----------+------------+---------+----------------------------+------------+--------------+------------+-----------------+---------+---------------+-------------+------------------+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------+----------------+-------------------+-----------+------------+
|PICKUP_DATETIME    |DROPOFF_DATETIME   |PU_LOCATION_ID|PU_ZONE            |PU_BOROUGH|DO_LOCATION_ID|DO_ZONE              |DO_BOROUGH|SERVICE_TYPE|VENDOR_ID|VENDOR_NAME                 |RATE_CODE_ID|RATE_CODE_DESC|PAYMENT_TYPE|PAYMENT_TYPE_DESC|TRIP_TYPE|PASSENGER_COUNT|TRIP_DISTANCE|STORE_AND_FWD_FLAG|FARE_AMOUNT|EXTRA|MTA_TAX|TIP_AMOUNT|TOLLS_AMOUNT|IMPROVEMENT_SURCHARGE|CONGESTION_SURCHARGE|AIRPORT_FEE|TOTAL_AMOUNT|RUN_ID          |INGESTED_AT_UTC    |SOURCE_YEAR|SOURCE_MONTH|
+-----